In [ ]:
from google.colab import auth; auth.authenticate_user()
from typing import Dict, Any, List, Protocol, Optional

import vertexai
client = vertexai.Client(project="qwiklabs-gcp-04-994305c355a7", location="us-central1")


# --- Define a Protocol for Agents ---
# This allows us to ensure all agents (root and sub) adhere to a common interface
class Agent(Protocol):
    """
    A protocol defining the common interface for all agents in the system.
    Agents should be able to describe themselves and process tasks.
    """
    name: str
    description: str

    def describe_capabilities(self) -> str:
        """
        Returns a human-readable description of what this agent can do.
        """
        ...

    def process_task(self, task_description: str, context: Dict[str, Any]) -> Dict[str, Any]:
        """
        Processes a given task description using its capabilities and updates
        or returns a context dictionary.
        """
        ...

# --- Example Sub-Agents ---

class WeatherAgent:
    """
    A sub-agent responsible for retrieving weather information.
    """
    name: str = "Weather Agent"
    description: str = "Retrieves current weather conditions and forecasts for specified locations."

    def describe_capabilities(self) -> str:
        return (f"{self.name}: {self.description}\n"
                "Capabilities: fetch_current_weather(location: str), fetch_forecast(location: str, days: int)")

    def process_task(self, task_description: str, context: Dict[str, Any]) -> Dict[str, Any]:
        """
        Processes a weather-related task.
        In a real scenario, this would call an external API (like NWS or a weather service).
        """
        print(f"WeatherAgent: Processing task '{task_description}'")
        location = context.get('location')
        if not location:
            return {"status": "failed", "reason": "No location specified for weather task."}

        # Simulate API call
        if "current weather" in task_description.lower():
            current_temp = "72°F and sunny" # Simulated
            context[f'weather_current_{location}'] = current_temp
            return {"status": "success", "message": f"Current weather in {location}: {current_temp}"}
        elif "forecast" in task_description.lower():
            forecast = "Partly cloudy with a high of 75°F for the next 3 days." # Simulated
            context[f'weather_forecast_{location}'] = forecast
            return {"status": "success", "message": f"Forecast for {location}: {forecast}"}
        else:
            return {"status": "failed", "reason": "Weather task not recognized."}

class SearchAgent:
    """
    A sub-agent responsible for performing web searches.
    """
    name: str = "Search Agent"
    description: str = "Performs web searches to find information on various topics."

    def describe_capabilities(self) -> str:
        return (f"{self.name}: {self.description}\n"
                "Capabilities: perform_search(query: str)")

    def process_task(self, task_description: str, context: Dict[str, Any]) -> Dict[str, Any]:
        """
        Processes a search-related task.
        In a real scenario, this would call a search engine API (e.g., Google Search API).
        """
        print(f"SearchAgent: Processing task '{task_description}'")
        query = context.get('search_query')
        if not query:
            return {"status": "failed", "reason": "No search query specified."}

        # Simulate search result
        search_result = f"Top result for '{query}': Example content from a web page." # Simulated
        context[f'search_results_{query}'] = search_result
        return {"status": "success", "message": f"Search for '{query}' completed. Found: {search_result}"}

# --- The Root Agent ---

class RootAgent:
    """
    The main coordinator agent that describes its overall capabilities and
    delegates tasks to specialized sub-agents.
    """
    name: str = "Coordinator Agent"
    description: str = "Orchestrates tasks, manages context, and delegates to specialized sub-agents " \
                       "to fulfill user requests. Can answer general knowledge questions, " \
                       "provide weather updates, and perform web searches."
    sub_agents: Dict[str, Agent]

    def __init__(self, sub_agents: List[Agent]):
        """
        Initializes the RootAgent with a list of available sub-agents.
        """
        self.sub_agents = {agent.name.lower(): agent for agent in sub_agents}
        print(f"RootAgent initialized with sub-agents: {', '.join(self.sub_agents.keys())}")

    def describe_capabilities(self) -> str:
        """
        Returns a comprehensive description of what the root agent and its
        sub-agents can achieve.
        """
        capabilities_str = f"I am the {self.name}, {self.description}\n\n"
        capabilities_str += "My sub-agents offer the following capabilities:\n"
        for agent_name, agent in self.sub_agents.items():
            capabilities_str += f"- {agent.describe_capabilities()}\n"
        return capabilities_str

    def _determine_sub_agent(self, task_description: str) -> Optional[Agent]:
        """
        Internal method to determine which sub-agent is best suited for a task.
        This would typically involve natural language understanding (NLU),
        keyword matching, or even a smaller LLM call for routing.
        """
        task_lower = task_description.lower()
        if "weather" in task_lower or "temperature" in task_lower or "forecast" in task_lower:
            return self.sub_agents.get("weather agent")
        elif "search" in task_lower or "find information" in task_lower or "look up" in task_lower:
            return self.sub_agents.get("search agent")
        # Add more sophisticated routing logic here
        return None

    def process_request(self, user_request: str) -> Dict[str, Any]:
        """
        Receives a user request, breaks it down (if necessary), delegates to
        appropriate sub-agents, and synthesizes a response.
        """
        print(f"\nRootAgent: Processing user request: '{user_request}'")
        context: Dict[str, Any] = {"original_request": user_request}
        final_response: Dict[str, Any] = {"status": "processing", "result": None}

        # Step 1: Initial understanding and task identification
        # In a real LLM-powered agent, an LLM would parse the request
        # to extract intent, entities, and required sub-tasks.
        if "weather in" in user_request.lower():
            parts = user_request.lower().split("weather in")
            if len(parts) > 1:
                location = parts[1].strip().split(" ")[0].replace("?", "").replace(".", "")
                context['location'] = location
                task_to_delegate = f"Get current weather and forecast for {location}"
                chosen_agent = self._determine_sub_agent(task_to_delegate)
            else:
                chosen_agent = None # Fallback or LLM to ask for more info
        elif "search for" in user_request.lower():
            query = user_request.lower().split("search for")[1].strip().replace("?", "").replace(".", "")
            context['search_query'] = query
            task_to_delegate = f"Perform web search for '{query}'"
            chosen_agent = self._determine_sub_agent(task_to_delegate)
        else:
            # For unhandled direct commands, try to route generically or respond directly
            chosen_agent = self._determine_sub_agent(user_request)
            task_to_delegate = user_request # Pass the whole request as a task

        if chosen_agent:
            print(f"RootAgent: Delegating to {chosen_agent.name} for task: '{task_to_delegate}'")
            agent_result = chosen_agent.process_task(task_to_delegate, context)
            if agent_result.get("status") == "success":
                final_response["status"] = "completed"
                final_response["result"] = agent_result.get("message")
                final_response["context_updates"] = context # Show how context evolved
            else:
                final_response["status"] = "failed_sub_agent"
                final_response["result"] = agent_result.get("reason", "Sub-agent failed.")
        else:
            # If no specific sub-agent found, the root agent might try to answer directly
            # or indicate it can't handle the request.
            if "what can you do" in user_request.lower():
                final_response["status"] = "completed"
                final_response["result"] = self.describe_capabilities()
            else:
                final_response["status"] = "failed"
                final_response["result"] = "I don't have a sub-agent equipped to handle that specific request directly."
                print("RootAgent: No suitable sub-agent found or task not recognized.")

        return final_response

# --- Main Execution ---
if __name__ == "__main__":
    # Instantiate sub-agents
    weather_agent = WeatherAgent()
    search_agent = SearchAgent()

    # Instantiate the root agent with its sub-agents
    root_agent = RootAgent(sub_agents=[weather_agent, search_agent])

    # Test the describe_capabilities method
    print("--- Agent Capabilities ---")
    print(root_agent.describe_capabilities())

    # Test with various user requests
    print("\n--- Processing User Requests ---")

    # Request 1: Get weather
    response1 = root_agent.process_request("What is the current weather in New York?")
    print(f"Response 1: {response1}\n")

    # Request 2: Get forecast
    response2 = root_agent.process_request("Can you give me the forecast for London for tomorrow?")
    print(f"Response 2: {response2}\n")

    # Request 3: Perform a search
    response3 = root_agent.process_request("Please search for information about large language models.")
    print(f"Response 3: {response3}\n")

    # Request 4: Ask about capabilities
    response4 = root_agent.process_request("What can you do?")
    print(f"Response 4: {response4}\n")

    # Request 5: Unhandled request
    response5 = root_agent.process_request("Tell me a funny joke.")
    print(f"Response 5: {response5}\n")

    # Test context persistence/updates (conceptual)
    # If the root agent had more complex planning, it could use context from one agent
    # for another. E.g., "What's the weather in the city where Google's main HQ is?"
    # -> Search for Google HQ -> Extract city -> Pass to Weather Agent.


RootAgent initialized with sub-agents: weather agent, search agent
--- Agent Capabilities ---
I am the Coordinator Agent, Orchestrates tasks, manages context, and delegates to specialized sub-agents to fulfill user requests. Can answer general knowledge questions, provide weather updates, and perform web searches.

My sub-agents offer the following capabilities:
- Weather Agent: Retrieves current weather conditions and forecasts for specified locations.
Capabilities: fetch_current_weather(location: str), fetch_forecast(location: str, days: int)
- Search Agent: Performs web searches to find information on various topics.
Capabilities: perform_search(query: str)


--- Processing User Requests ---

RootAgent: Processing user request: 'What is the current weather in New York?'
RootAgent: Delegating to Weather Agent for task: 'Get current weather and forecast for new'
WeatherAgent: Processing task 'Get current weather and forecast for new'
Response 1: {'status': 'completed', 'result': 'Curr

In [93]:
print(f"\n--- Interacting with the deployed RootAgent (ReasoningEngine) ---")

# The remote_root_agent object was created in the previous cell.
# If you are running this in a new session, you would load it like this:
# remote_root_agent = vertexai.preview.reasoning_engines.ReasoningEngine.load(remote_root_agent_resource_name)

# Example 1: Send a weather request
request_1 = {"user_input": "What's the weather like in New York?"}
response_re_1 = remote_root_agent.query(**request_1) # Unpack the dictionary
print(f"\nReasoning Engine Response 1: {response_re_1}")

# Example 2: Send a search request
request_2 = {"user_input": "Search for information about Google Cloud Platform."}
response_re_2 = remote_root_agent.query(**request_2) # Unpack the dictionary
print(f"\nReasoning Engine Response 2: {response_re_2}")

# Example 3: Send a question answering request
request_3 = {"user_input": "Who is Albert Einstein?"}
response_re_3 = remote_root_agent.query(**request_3) # Unpack the dictionary
print(f"\nReasoning Engine Response 3: {response_re_3}")

# Example 4: Send a route request
request_4 = {"user_input": "Get me a route from London to Paris by driving"}
response_re_4 = remote_root_agent.query(**request_4) # Unpack the dictionary
print(f"\nReasoning Engine Response 4: {response_re_4}")


--- Interacting with the deployed RootAgent (ReasoningEngine) ---

Reasoning Engine Response 1: {'result': 'Current conditions in new york: 37°F and Light Rain. Detailed: Rain before 7pm, then rain and patchy fog. Mostly cloudy. Low around 37, with temperatures rising to around 41 overnight. West wind 5 to 9 mph. Chance of precipitation is 100%. New rainfall amounts between a quarter and half of an inch possible.', 'status': 'completed', 'context_updates': {'weather_new york': 'Current conditions in new york: 37°F and Light Rain. Detailed: Rain before 7pm, then rain and patchy fog. Mostly cloudy. Low around 37, with temperatures rising to around 41 overnight. West wind 5 to 9 mph. Chance of precipitation is 100%. New rainfall amounts between a quarter and half of an inch possible.', 'location': 'new york', 'original_request': "What's the weather like in New York?"}}

Reasoning Engine Response 2: {'result': "Search for 'information about google cloud platform' completed. Top result: Si

In [86]:
import requests
import json
from typing import Dict, Any, List, Protocol, Optional
import re # Import re module

# --- Define a Protocol for Agents ---
class Agent(Protocol):
    """
    A protocol defining the common interface for all agents in the system.
    Agents should be able to describe themselves and process tasks.
    """
    name: str
    description: str

    def describe_capabilities(self) -> str:
        """
        Returns a human-readable description of what this agent can do.
        """
        ...

    def process_task(self, task_description: str, context: Dict[str, Any]) -> Dict[str, Any]:
        """
        Processes a given task description using its capabilities and updates
        or returns a context dictionary.
        Returns a dictionary with at least 'status' (success/failed) and 'message'.
        """
        ...

# --- Sub-Agent Implementations ---

class WeatherAgent:
    """
    A sub-agent responsible for retrieving weather information using the NWS API.
    """
    name: str = "Weather Agent"
    description: str = "Retrieves current weather conditions and forecasts for specified locations using the National Weather Service (NWS) API."

    def describe_capabilities(self) -> str:
        return (f"{self.name}: {self.description}\n"
                "Capabilities: fetch_weather(location: str) for current conditions and forecast.")

    def process_task(self, task_description: str, context: Dict[str, Any]) -> Dict[str, Any]:
        """
        Processes a weather-related task for a given location.
        Requires 'location' in the context.
        """
        print(f"[{self.name}] Processing task: '{task_description}'")
        location_name = context.get('location')

        if not location_name:
            return {"status": "failed", "message": "No location specified for weather query."}

        # In a real scenario, you'd need a more robust way to convert
        # a location name to lat/lon, e.g., a geocoding service.
        # For this example, we'll use hardcoded values for a few cities.
        locations = {
            "new york": {"lat": 40.7128, "lon": -74.0060},
            "london": {"lat": 51.5074, "lon": -0.1278},
            "tokyo": {"lat": 35.6895, "lon": 139.6917},
            "paris": {"lat": 48.8566, "lon": 2.3522},
            # Default or fallback
            "default": {"lat": 34.0522, "lon": -118.2437} # Los Angeles
        }
        coords = locations.get(location_name.lower(), locations["default"])
        latitude = coords["lat"]
        longitude = coords["lon"]

        try:
            # Step 1: Get the forecast office and grid points from NWS
            points_url = f"https://api.weather.gov/points/{latitude},{longitude}"
            headers = {'User-Agent': 'MyMultiAgentSystem/1.0 (your_email@example.com)'} # NWS requires User-Agent
            points_response = requests.get(points_url, headers=headers, timeout=5)
            points_response.raise_for_status()
            points_data = points_response.json()

            forecast_url = points_data['properties']['forecast']

            # Step 2: Get the detailed forecast
            forecast_response = requests.get(forecast_url, headers=headers, timeout=5)
            forecast_response.raise_for_status()
            forecast_data = forecast_response.json()

            # Extract relevant info for the current or next period
            periods = forecast_data['properties']['periods']
            if periods:
                current_period = periods[0]
                summary = f"Current conditions in {location_name}: {current_period.get('temperature')}°{current_period.get('temperatureUnit')} and {current_period.get('shortForecast')}. Detailed: {current_period.get('detailedForecast')}"
                context[f'weather_{location_name}'] = summary
                return {"status": "success", "message": summary}
            else:
                return {"status": "failed", "message": f"No forecast periods found for {location_name}."}

        except requests.exceptions.RequestException as e:
            return {"status": "failed", "message": f"Could not retrieve weather for {location_name}: {e}"}
        except KeyError as e:
            return {"status": "failed", "message": f"Error parsing weather data for {location_name} (missing key: {e})."}
        except Exception as e:
            return {"status": "failed", "message": f"An unexpected error occurred: {e}"}


class InternetSearchAgent:
    """
    A sub-agent responsible for performing internet searches.
    (Placeholder: In a real system, this would integrate with a search API like Google Custom Search, SerpApi, etc.)
    """
    name: str = "Internet Search Agent"
    description: str = "Performs web searches to find information on various topics and retrieves relevant snippets."

    def describe_capabilities(self) -> str:
        return (f"{self.name}: {self.description}\n"
                "Capabilities: perform_web_search(query: str) to get top search results.")

    def process_task(self, task_description: str, context: Dict[str, Any]) -> Dict[str, Any]:
        """
        Processes a search task. Requires 'search_query' in the context.
        """
        print(f"[{self.name}] Processing task: '{task_description}'")
        query = context.get('search_query')

        if not query:
            return {"status": "failed", "message": "No search query provided."}

        # Simulate fetching search results
        # In a real scenario, you'd use a search API like this:
        # SEARCH_API_KEY = "YOUR_GOOGLE_CUSTOM_SEARCH_API_KEY"
        # CX_ID = "YOUR_CUSTOM_SEARCH_ENGINE_ID"
        # search_url = f"https://www.googleapis.com/customsearch/v1?key={SEARCH_API_KEY}&cx={CX_ID}&q={query}"
        # response = requests.get(search_url).json()
        # top_result = response.get('items', [{}])[0].get('snippet', 'No result found.')

        simulated_results = {
            "google cloud": "Google Cloud Platform (GCP) is a suite of cloud computing services that runs on the same infrastructure that Google uses internally for its end-user products.",
            "large language models": "Large language models (LLMs) are deep learning models that can recognize, summarize, translate, predict, and generate text and other content based on knowledge gained from massive datasets.",
            "vertex ai": "Vertex AI is Google Cloud's managed machine learning platform for building, deploying, and scaling ML models.",
            "default": f"Simulated search result for '{query}': Information suggests that '{query}' is a relevant topic in today's news."
        }

        top_result = simulated_results.get(query.lower(), simulated_results["default"])
        context[f'search_result_for_{query}'] = top_result
        return {"status": "success", "message": f"Search for '{query}' completed. Top result: {top_result}"}


class GoogleMapsAgent:
    """
    A sub-agent responsible for providing routes and navigation information
    using the Google Maps Platform APIs.
    (Placeholder: This requires a Google Maps API Key and actual API calls.)
    """
    name: str = "Google Maps Agent"
    description: str = "Provides routes, travel times, and navigation instructions between locations using Google Maps Platform APIs."
    google_maps_api_key: Optional[str] = None # Placeholder for actual key

    def __init__(self, api_key: Optional[str] = None):
        self.google_maps_api_key = api_key or "YOUR_GOOGLE_MAPS_API_KEY" # Replace with your actual key

    def describe_capabilities(self) -> str:
        return (f"{self.name}: {self.description}\n"
                "Capabilities: get_route(origin: str, destination: str, mode: str = 'driving') for directions.")

    def process_task(self, task_description: str, context: Dict[str, Any]) -> Dict[str, Any]:
        """
        Processes a route finding task. Requires 'origin' and 'destination' in the context.
        """
        print(f"[{self.name}] Processing task: '{task_description}'")
        origin = context.get('origin')
        destination = context.get('destination')
        travel_mode = context.get('travel_mode', 'driving') # Default to driving

        if not origin or not destination:
            return {"status": "failed", "message": "Origin and destination must be provided for routing."}

        if self.google_maps_api_key == "YOUR_GOOGLE_MAPS_API_KEY" or not self.google_maps_api_key:
            return {"status": "failed", "message": "Google Maps API Key not configured. Cannot get route."}

        # --- Simulated Google Maps API Call ---
        # In a real scenario, you'd make an HTTP request to the Directions API:
        # url = f"https://maps.googleapis.com/maps/api/directions/json?origin={origin}&destination={destination}&mode={travel_mode}&key={self.google_maps_api_key}"
        # try:
        #     response = requests.get(url, timeout=5).json()
        #     if response['status'] == 'OK' and response['routes']:
        #         route_summary = response['routes'][0]['summary']
        #         leg = response['routes'][0]['legs'][0]
        #         distance = leg['distance']['text']
        #         duration = leg['duration']['text']
        #         message = f"Route from {origin} to {destination} ({travel_mode}): {route_summary}. Distance: {distance}, Duration: {duration}."
        #         context['route_info'] = message
        #         return {"status": "success", "message": message}
        #     else:
        #         return {"status": "failed", "message": f"Could not find route: {response.get('error_message', 'No route found.')}"}
        # except requests.exceptions.RequestException as e:
        #     return {"status": "failed", "message": f"Google Maps API request failed: {e}"}

        # For now, using simulated data:
        simulated_route_info = {
            "driving": f"Simulated driving route from {origin} to {destination}: Take the main highway, estimated 30 minutes.",
            "walking": f"Simulated walking route from {origin} to {destination}: Approximately 2 hours, through parks and residential areas.",
            "bicycling": f"Simulated bicycling route from {origin} to {destination}: Scenic route, about 1 hour 15 minutes.",
            "transit": f"Simulated transit route from {origin} to {destination}: Take Line A, then transfer to Bus 7, estimated 45 minutes."
        }
        route_message = simulated_route_info.get(travel_mode.lower(), f"Simulated route for {origin} to {destination}: Generic directions.")
        context['route_info'] = route_message
        return {"status": "success", "message": route_message}


class QuestionAnsweringAgent:
    """
    A sub-agent for answering general knowledge questions.
    (Placeholder: In a real system, this would be powered by a Large Language Model (LLM)
    like Google's Gemini via Vertex AI, or a knowledge base lookup.)
    """
    name: str = "Question Answering Agent"
    description: str = "Answers general knowledge questions based on its training data or integrated knowledge sources."

    def describe_capabilities(self) -> str:
        return (f"{self.name}: {self.description}\n"
                "Capabilities: answer_question(query: str) to get factual information.")

    def process_task(self, task_description: str, context: Dict[str, Any]) -> Dict[str, Any]:
        """
        Processes a question-answering task. Requires 'question' in the context.
        """
        print(f"[{self.name}] Processing task: '{task_description}'")
        question = context.get('question')

        if not question:
            return {"status": "failed", "message": "No question provided to answer."}

        # --- Simulated Question Answering Logic ---
        # In a real setup, this would be an LLM call:
        # from vertexai.generative_models import GenerativeModel
        # model = GenerativeModel("gemini-pro")
        # response = model.generate_content(question)
        # answer = response.text

        answers = {
            "capital of france": "The capital of France is Paris.",
            "largest ocean": "The Pacific Ocean is the largest and deepest of Earth's oceanic divisions.",
            "who is albert einstein": "Albert Einstein was a German-born theoretical physicist who developed the theory of relativity, one of the two pillars of modern physics.",
            "what is the meaning of life": "The meaning of life is a philosophical question concerning the significance of living or existence in general. Different cultures and ideologies offer a wide range of answers.",
            "default": f"I'm sorry, I can't answer '{question}' with my current knowledge base."
        }
        answer = answers.get(question.lower().replace("?", "").strip(), answers["default"])

        context[f'answer_to_{question}'] = answer
        return {"status": "success", "message": answer}


# --- The Root Agent (Coordinator) ---

class RootAgent:
    """
    The main coordinator agent that describes its overall capabilities and
    delegates tasks to specialized sub-agents.
    """
    name: str = "Coordinator Agent"
    description: str = "Orchestrates tasks, manages context, and delegates to specialized sub-agents " \
                       "to fulfill user requests. Can provide weather forecasts, search the internet, " \
                       "offer routes, and answer general questions."
    sub_agents: Dict[str, Agent]

    def __init__(self, sub_agents: List[Agent]):
        """
        Initializes the RootAgent with a list of available sub-agents.
        """
        self.sub_agents = {agent.name.lower(): agent for agent in sub_agents}
        print(f"RootAgent initialized with sub-agents: {', '.join(self.sub_agents.keys())}")

    def describe_capabilities(self) -> str:
        """
        Returns a comprehensive description of what the root agent and its
        sub-agents can achieve.
        """
        capabilities_str = f"I am the {self.name}, {self.description}\n\n"
        capabilities_str += "My sub-agents offer the following capabilities:\n"
        for agent_name, agent in self.sub_agents.items():
            capabilities_str += f"- {agent.describe_capabilities()}\n"
        return capabilities_str

    def _determine_sub_agent_and_extract_params(self, user_request: str) -> Optional[Dict[str, Any]]:
        """
        Internal method to determine which sub-agent is best suited for a task
        and extract relevant parameters. This is where an LLM would shine.
        """
        user_request_lower = user_request.lower()

        # Priority 1: Direct "what can you do"
        if "what can you do" in user_request_lower:
            return {"agent_name": "self_description", "parameters": {}}

        # Priority 2: Google Maps
        if "route" in user_request_lower or "directions" in user_request_lower:
            origin = None
            destination = None
            mode = "driving" # Default
            # Very basic extraction. LLM would be much better.
            if "from" in user_request_lower and "to" in user_request_lower:
                parts = user_request_lower.split("from")
                if len(parts) > 1:
                    origin_part = parts[1].split("to")[0].strip()
                    origin = origin_part.split("using")[0].split("by")[0].strip()
                    destination_part = parts[1].split("to")[1].strip()
                    destination = destination_part.split("using")[0].split("by")[0].strip()
            if "walking" in user_request_lower: mode = "walking"
            elif "transit" in user_request_lower: mode = "transit"
            elif "bicycling" in user_request_lower: mode = "bicycling"

            if origin and destination:
                return {"agent_name": "google maps agent", "parameters": {"origin": origin, "destination": destination, "travel_mode": mode}}

        # Priority 3: Weather
        if "weather" in user_request_lower or "forecast" in user_request_lower or "temperature" in user_request_lower:
            location = None
            # Try to extract location after common weather phrases using regex
            match = re.search(r"(?:weather (?:like )?in|forecast for|temperature in)\s+([a-zA-Z\s]+?)(?:\?|\.|,|$)", user_request_lower)
            if match:
                location = match.group(1).strip()

            if location:
                return {"agent_name": "weather agent", "parameters": {"location": location}}

        # Priority 4: Search
        if "search for" in user_request_lower or "look up" in user_request_lower or "find information about" in user_request_lower:
            query = None
            for phrase in ["search for ", "look up ", "find information about "]:
                if phrase in user_request_lower:
                    query = user_request_lower.split(phrase, 1)[1].split("?", 1)[0].split(".", 1)[0].strip()
                    break
            if query:
                return {"agent_name": "internet search agent", "parameters": {"search_query": query}}

        # Priority 5: Question Answering (fallback for general questions)
        # This is a very broad category; an LLM here would be essential.
        if user_request_lower.startswith(("what is", "who is", "where is", "when is", "how is", "tell me about")):
            question = user_request_lower.replace("what is", "").replace("who is", "").replace("where is", "").replace("when is", "").replace("how is", "").replace("tell me about", "").strip()
            if question:
                return {"agent_name": "question answering agent", "parameters": {"question": question}}


        return None # No suitable agent found

    def process_request(self, user_request: str) -> Dict[str, Any]:
        """
        Receives a user request, breaks it down (if necessary), delegates to
        appropriate sub-agents, and synthesizes a response.
        """
        print(f"\n[{self.name}] Processing user request: '{user_request}'")
        context: Dict[str, Any] = {"original_request": user_request}
        final_response: Dict[str, Any] = {"status": "processing", "result": None}

        # Step 1: Determine agent and extract parameters
        routing_info = self._determine_sub_agent_and_extract_params(user_request)

        if routing_info and routing_info.get("agent_name") == "self_description":
            final_response["status"] = "completed"
            final_response["result"] = self.describe_capabilities()
            return final_response

        if routing_info and routing_info.get("agent_name"):
            agent_name_lower = routing_info["agent_name"].lower()
            chosen_agent = self.sub_agents.get(agent_name_lower)
            parameters = routing_info.get("parameters", {})

            if chosen_agent:
                context.update(parameters) # Add extracted parameters to context
                task_description = f"Handle: {user_request}" # Generic task description for sub-agent
                print(f"[{self.name}] Delegating to {chosen_agent.name} with parameters: {parameters}")
                agent_result = chosen_agent.process_task(task_description, context)

                if agent_result.get("status") == "success":
                    final_response["status"] = "completed"
                    final_response["result"] = agent_result.get("message")
                    final_response["context_updates"] = context # Show how context evolved
                else:
                    final_response["status"] = "failed_sub_agent"
                    final_response["result"] = agent_result.get("message", "Sub-agent failed.")
            else:
                final_response["status"] = "failed"
                final_response["result"] = f"Internal error: Could not find agent '{agent_name_lower}'."
        else:
            final_response["status"] = "failed"
            final_response["result"] = "I don't have an agent equipped to handle that specific request or couldn't parse it."
            print(f"[{self.name}] No suitable sub-agent found or task not recognized for: '{user_request}'")

        return final_response

# --- Global Initialization for Agent Deployment ---
# Initialize sub-agents
weather_agent = WeatherAgent()
search_agent = InternetSearchAgent()
maps_agent = GoogleMapsAgent(api_key="YOUR_ACTUAL_GOOGLE_MAPS_API_KEY") # Replace with your actual key or use environment variables
qa_agent = QuestionAnsweringAgent()

# Instantiate the root agent with its sub-agents
root_agent = RootAgent(sub_agents=[weather_agent, search_agent, maps_agent, qa_agent])

def process_agent_request(user_request: str) -> Dict[str, Any]:
    """
    Entry point for the agent engine to process a user request.
    """
    return root_agent.process_request(user_request)

# Example of how to call the entry point (for testing purposes, not part of deployed agent logic)
# if __name__ == "__main__":
#     print("--- Agent Capabilities ---")
#     print(root_agent.describe_capabilities())

#     print("\n--- Processing User Requests ---")
#     response = process_agent_request("What's the weather like in New York?")
#     print(f"Agent Response: {response}\n")


RootAgent initialized with sub-agents: weather agent, internet search agent, google maps agent, question answering agent


In [ ]:
from typing import Dict, Any, List, Optional, Callable
import json # For structured output

# Assume Agent, WeatherAgent, InternetSearchAgent, GoogleMapsAgent, QuestionAnsweringAgent, RootAgent
# are defined as in the previous response.
# For brevity, I'm omitting their full re-definition here, but they would be imported or
# included directly in a real script.
# You'd need the full code from the previous response for this to run.

# --- Define a Protocol for Workflow Steps (Validator/Refiner) ---
class WorkflowStep(Protocol):
    """
    A protocol defining the interface for a single step in a sequential workflow.
    Each step takes a result and context, and either validates or refines it.
    """
    name: str
    description: str

    def process(self, input_data: Dict[str, Any], context: Dict[str, Any]) -> Dict[str, Any]:
        """
        Processes the input data (e.g., agent's response) and context.
        Returns an updated dictionary with at least 'status' (success/failed) and 'output'.
        'output' can be the validated/refined data or an error message.
        """
        ...

# --- Workflow Step Implementations ---

class ResponseValidator:
    """
    A workflow step that validates the structure and basic content of an agent's response.
    This step could be powered by an LLM for more complex semantic validation.
    """
    name: str = "Response Validator"
    description: str = "Checks if the agent's response is well-formed, complete, and relevant to the original request."

    def process(self, input_data: Dict[str, Any], context: Dict[str, Any]) -> Dict[str, Any]:
        """
        Validates the 'result' field of the input_data (assumed to be from an agent).
        """
        print(f"[{self.name}] Validating response...")
        agent_result = input_data.get('result')
        original_request = context.get('original_request')

        if not agent_result:
            return {"status": "failed", "output": "Agent produced an empty or missing result."}

        # Basic length check
        if len(agent_result) < 10:
            return {"status": "failed", "output": "Agent response is too short and potentially incomplete."}

        # Basic relevance check (can be greatly enhanced by an LLM)
        if original_request and "weather" in original_request.lower():
            if "temperature" not in agent_result.lower() and "forecast" not in agent_result.lower():
                return {"status": "failed", "output": "Weather response does not contain typical weather keywords."}
        elif original_request and "search" in original_request.lower():
             if "search result" not in agent_result.lower(): # Simple check
                 return {"status": "failed", "output": "Search response does not seem to contain search results."}

        # If all checks pass
        return {"status": "success", "output": agent_result}

class ResponseRefiner:
    """
    A workflow step that refines an agent's response for clarity, conciseness,
    and better formatting for the end-user. This is an ideal place for an LLM.
    """
    name: str = "Response Refiner"
    description: str = "Rephrases and formats the validated response to be more user-friendly and concise."

    def process(self, input_data: Dict[str, Any], context: Dict[str, Any]) -> Dict[str, Any]:
        """
        Refines the 'output' field of the input_data (assumed to be validated result).
        """
        print(f"[{self.name}] Refining response...")
        validated_output = input_data.get('output')
        original_request = context.get('original_request', 'your request')

        if not validated_output:
            return {"status": "failed", "output": "No valid output to refine."}

        # --- LLM Call for Refinement ---
        # This is where you'd typically call a powerful LLM (like Gemini via Vertex AI)
        # to rephrase and improve the response.
        #
        # For example, using the Vertex AI SDK:
        # from vertexai.generative_models import GenerativeModel
        # model = GenerativeModel("gemini-pro")
        # prompt = f"""
        # The user asked: "{original_request}"
        # An agent provided this raw response: "{validated_output}"
        # Please rephrase this response to be more concise, natural, and directly answer the user's question.
        # Ensure it's suitable for a conversational AI.
        # """
        # llm_response = model.generate_content(prompt).text
        # refined_response = llm_response
        #
        # For this example, we'll use a simple rule-based refiner:

        refined_response = f"Regarding '{original_request}': {validated_output.strip().replace('Detailed: ', 'In detail: ')}"
        if "simulated driving route" in refined_response.lower():
            refined_response = refined_response.replace("Simulated driving route from", "Here's a simulated driving route from")
        if "simulated search result" in refined_response.lower():
            refined_response = refined_response.replace("Simulated search result for", "I found this information for")

        return {"status": "success", "output": refined_response}

class ResponseSummarizer:
    """
    A workflow step that summarizes longer responses, especially useful for multi-step tasks.
    This is also an excellent candidate for an LLM.
    """
    name: str = "Response Summarizer"
    description: str = "Summarizes lengthy responses into key takeaways, typically for conversational interaction."

    def process(self, input_data: Dict[str, Any], context: Dict[str, Any]) -> Dict[str, Any]:
        """
        Summarizes the 'output' field if it's too long.
        """
        print(f"[{self.name}] Summarizing response (if needed)...")
        refined_output = input_data.get('output')

        if not refined_output:
            return {"status": "failed", "output": "No refined output to summarize."}

        # Simple length-based summarization (LLM would be better)
        if len(refined_output) > 200 and not context.get('is_detailed_request', False):
            # Example: "Regarding 'What's the weather like in New York?': Current conditions in New York: 72°F and sunny and Partly cloudy. In detail: Partly cloudy with a high of 75°F for the next 3 days."
            # Find a good place to truncate
            summary_parts = refined_output.split(". ")
            if len(summary_parts) > 1:
                summary = summary_parts[0] + "."
                if len(summary) < len(refined_output):
                    print(f"[{self.name}] Response summarized.")
                    return {"status": "success", "output": summary}
        return {"status": "success", "output": refined_output} # Return as is if no summarization needed


class SequentialWorkflow:
    """
    Orchestrates a sequence of workflow steps (validation, refinement, summarization)
    on the initial output from an agent.
    """
    name: str = "Sequential Workflow Processor"
    steps: List[WorkflowStep]

    def __init__(self, steps: List[WorkflowStep]):
        self.steps = steps

    def execute(self, agent_output: Dict[str, Any], context: Dict[str, Any]) -> Dict[str, Any]:
        """
        Executes the predefined sequence of steps.
        """
        current_data = agent_output.copy() # Start with the agent's raw output

        print(f"\n--- {self.name} started ---")
        for step in self.steps:
            print(f"  Executing step: {step.name}")
            processed_data = step.process(current_data, context)

            if processed_data.get("status") == "failed":
                print(f"  [{step.name}] Failed: {processed_data.get('output')}")
                return {"status": "failed", "final_output": processed_data.get('output', "Workflow failed at validation/refinement step.")}
            else:
                current_data["result"] = processed_data.get("output") # Update the 'result' for next step
                current_data["output"] = processed_data.get("output") # Keep consistent key naming

        print(f"--- {self.name} finished ---")
        return {"status": "completed", "final_output": current_data.get("output")}


# --- Main Execution (integrating with previous RootAgent) ---
if __name__ == "__main__":
    # --- Setup Agents (from previous code) ---
    # For a runnable example, you'd need the full definitions of these classes.
    # I'm using placeholder references here.
    # You would uncomment and use the actual imports/definitions.
    # from previous_agents import WeatherAgent, InternetSearchAgent, GoogleMapsAgent, QuestionAnsweringAgent, RootAgent

    class MockAgent: # Simplified Mock for demonstration if full agents aren't present
        name: str = "Mock Agent"
        description: str = "A mock agent for testing workflows."
        def describe_capabilities(self) -> str: return self.description
        def process_task(self, task_description: str, context: Dict[str, Any]) -> Dict[str, Any]:
            # Simulate a successful agent response
            if "weather" in task_description:
                return {"status": "success", "message": "Raw weather data: T=72F, P=Sunny, F=Partly cloudy tomorrow. Detailed: Expect 75F tomorrow with light winds."}
            elif "search" in task_description:
                return {"status": "success", "message": "Raw search result: LLMs are deep learning models. They are trained on huge datasets of text. Applications include chatbots and content generation."}
            elif "joke" in task_description:
                return {"status": "success", "message": "Why don't scientists trust atoms? Because they make up everything!"}
            else:
                return {"status": "failed", "message": "Mock agent failed to understand task."}

    class MockRootAgent: # Simplified Mock
        name: str = "Mock Root Agent"
        description: str = "Mocks the RootAgent functionality."
        sub_agents: Dict[str, Any]
        def __init__(self, sub_agents: List[Any]): self.sub_agents = {a.name.lower(): a for a in sub_agents}
        def describe_capabilities(self) -> str: return "Mock capabilities."
        def process_request(self, user_request: str) -> Dict[str, Any]:
            # Simulate RootAgent output format
            context_data = {"original_request": user_request}
            if "weather" in user_request.lower():
                agent_result = self.sub_agents['mock agent'].process_task("weather task", context_data)
                return {"status": "completed", "result": agent_result['message'], "context_updates": context_data}
            elif "search" in user_request.lower():
                agent_result = self.sub_agents['mock agent'].process_task("search task", context_data)
                return {"status": "completed", "result": agent_result['message'], "context_updates": context_data}
            elif "joke" in user_request.lower():
                agent_result = self.sub_agents['mock agent'].process_task("joke task", context_data)
                return {"status": "completed", "result": agent_result['message'], "context_updates": context_data}
            return {"status": "failed", "result": "No mock agent found for this request.", "context_updates": context_data}


    # --- Initialize Actual/Mock Agents ---
    # For a real run, uncomment these and provide real API keys for GoogleMapsAgent
    # weather_agent = WeatherAgent()
    # search_agent = InternetSearchAgent()
    # maps_agent = GoogleMapsAgent(api_key="YOUR_ACTUAL_GOOGLE_MAPS_API_KEY")
    # qa_agent = QuestionAnsweringAgent()
    # root_agent = RootAgent(sub_agents=[weather_agent, search_agent, maps_agent, qa_agent])

    # For this demonstration, we'll use a simple mock agent and root agent
    mock_sub_agent = MockAgent()
    root_agent = MockRootAgent(sub_agents=[mock_sub_agent])


    # --- Setup the Workflow ---
    workflow_steps: List[WorkflowStep] = [
        ResponseValidator(),
        ResponseRefiner(),
        ResponseSummarizer() # Optional: Add if summarization is often needed
    ]
    response_workflow = SequentialWorkflow(steps=workflow_steps)

    # --- Process requests through the RootAgent and then the Workflow ---
    print("\n--- Processing Request 1: Weather (Success) ---")
    user_request_1 = "What's the weather like in New York?"
    agent_raw_output_1 = root_agent.process_request(user_request_1)

    if agent_raw_output_1["status"] == "completed":
        workflow_result_1 = response_workflow.execute(agent_raw_output_1, agent_raw_output_1["context_updates"])
        print(f"Final Output 1: {workflow_result_1.get('final_output')}")
    else:
        print(f"Agent failed for Request 1: {agent_raw_output_1.get('result')}")

    print("\n--- Processing Request 2: Search (Success) ---")
    user_request_2 = "Search for information about Large Language Models."
    agent_raw_output_2 = root_agent.process_request(user_request_2)

    if agent_raw_output_2["status"] == "completed":
        workflow_result_2 = response_workflow.execute(agent_raw_output_2, agent_raw_output_2["context_updates"])
        print(f"Final Output 2: {workflow_result_2.get('final_output')}")
    else:
        print(f"Agent failed for Request 2: {agent_raw_output_2.get('result')}")

    print("\n--- Processing Request 3: Joke (Less suitable for summarization) ---")
    user_request_3 = "Tell me a joke."
    agent_raw_output_3 = root_agent.process_request(user_request_3)

    if agent_raw_output_3["status"] == "completed":
        workflow_result_3 = response_workflow.execute(agent_raw_output_3, agent_raw_output_3["context_updates"])
        print(f"Final Output 3: {workflow_result_3.get('final_output')}")
    else:
        print(f"Agent failed for Request 3: {agent_raw_output_3.get('result')}")


In [ ]:
from datetime import datetime

# --- Example Web Application Context (e.g., Flask, Django, etc.) ---

# This function simulates a request object that would be passed to your callback
class MockRequest:
    def __init__(self, user_agent_string):
        self.headers = {'User-Agent': user_agent_string}
        self.path = '/example/path'
        self.method = 'GET'

# --- Callback Function for Logging User-Agent ---

def log_user_agent_callback(request):
    """
    A callback function to log user-agent interactions.
    This function would typically be registered to run before or after
    processing each incoming web request.
    """
    user_agent = request.headers.get('User-Agent', 'N/A')
    timestamp = datetime.now().isoformat()
    log_message = (
        f"[{timestamp}] "
        f"Method: {request.method}, "
        f"Path: {request.path}, "
        f"User-Agent: '{user_agent}'"
    )
    print(f"LOG: {log_message}")
    # In a real application, you would write this to a file, a database,
    # or a logging service like Cloud Logging.

# --- How to integrate this in a hypothetical web application ---

# Example of how you might register and use the callback
def app_request_handler(request):
    # Call the logging callback before processing the request
    log_user_agent_callback(request)

    # --- Your application's core logic would go here ---
    # For demonstration, we'll just return a simple response
    return f"Hello from {request.path}! Your User-Agent was logged."

# --- Demonstration of usage ---
if __name__ == "__main__":
    print("--- Simulating incoming requests ---")

    # Simulate a request from a Chrome browser
    chrome_request = MockRequest(
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    response_chrome = app_request_handler(chrome_request)
    print(f"App Response: {response_chrome}\n")

    # Simulate a request from a mobile Safari browser
    safari_mobile_request = MockRequest(
        "Mozilla/5.0 (iPhone; CPU iPhone OS 17_3_1 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.3.1 Mobile/15E148 Safari/604.1"
    )
    response_safari = app_request_handler(safari_mobile_request)
    print(f"App Response: {response_safari}\n")

    # Simulate a request with no User-Agent header (or a bot)
    no_ua_request = MockRequest("") # Or you could omit the header entirely
    response_no_ua = app_request_handler(no_ua_request)
    print(f"App Response: {response_no_ua}\n")


In [ ]:
from typing import Optional

def validate_age(age_str: str) -> Optional[int]:
    """
    Validates if a string represents a valid age (integer between 0 and 120).

    Args:
        age_str: The string input representing the age.

    Returns:
        The validated age as an integer if valid, otherwise None.
    """
    try:
        age = int(age_str)
        if 0 <= age <= 120:
            return age
        else:
            print(f"Validation Error: Age {age} is out of the valid range (0-120).")
            return None
    except ValueError:
        print(f"Validation Error: '{age_str}' is not a valid integer for age.")
        return None

def validate_temperature(temp_str: str, min_val: float = -100.0, max_val: float = 100.0) -> Optional[float]:
    """
    Validates if a string represents a valid temperature (float within a specified range).

    Args:
        temp_str: The string input representing the temperature.
        min_val: The minimum acceptable temperature.
        max_val: The maximum acceptable temperature.

    Returns:
        The validated temperature as a float if valid, otherwise None.
    """
    try:
        temp = float(temp_str)
        if min_val <= temp <= max_val:
            return temp
        else:
            print(f"Validation Error: Temperature {temp} is out of the valid range ({min_val}-{max_val}).")
            return None
    except ValueError:
        print(f"Validation Error: '{temp_str}' is not a valid number for temperature.")
        return None

# --- Usage Examples ---
if __name__ == "__main__":
    print("--- Age Validation ---")
    age1 = validate_age("30")
    print(f"Validated age '30': {age1}")

    age2 = validate_age("-5")
    print(f"Validated age '-5': {age2}")

    age3 = validate_age("150")
    print(f"Validated age '150': {age3}")

    age4 = validate_age("abc")
    print(f"Validated age 'abc': {age4}")

    print("\n--- Temperature Validation ---")
    temp1 = validate_temperature("25.5")
    print(f"Validated temperature '25.5': {temp1}")

    temp2 = validate_temperature("120.0")
    print(f"Validated temperature '120.0': {temp2}")

    temp3 = validate_temperature("-150")
    print(f"Validated temperature '-150': {temp3}")

    temp4 = validate_temperature("not_a_number")
    print(f"Validated temperature 'not_a_number': {temp4}")


In [ ]:
import re
from typing import Optional

def validate_email(email: str) -> Optional[str]:
    """
    Validates if a string is a well-formed email address.

    Args:
        email: The email string to validate.

    Returns:
        The validated email string if valid, otherwise None.
    """
    # A common regex for email validation (can be more complex depending on strictness)
    # This regex is a good balance for general use.
    email_regex = r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$"
    if re.fullmatch(email_regex, email):
        return email
    else:
        print(f"Validation Error: '{email}' is not a valid email address.")
        return None

def validate_phone_number(phone: str) -> Optional[str]:
    """
    Validates if a string is a well-formed US phone number (e.g., XXX-XXX-XXXX).

    Args:
        phone: The phone number string to validate.

    Returns:
        The validated phone number string if valid, otherwise None.
    """
    # Example for US phone numbers (XXX-XXX-XXXX or (XXX) XXX-XXXX)
    phone_regex = r"^\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}$"
    if re.fullmatch(phone_regex, phone):
        return phone
    else:
        print(f"Validation Error: '{phone}' is not a valid US phone number format (e.g., XXX-XXX-XXXX).")
        return None

def validate_url(url: str) -> Optional[str]:
    """
    Validates if a string is a well-formed URL.

    Args:
        url: The URL string to validate.

    Returns:
        The validated URL string if valid, otherwise None.
    """
    # A comprehensive regex for URLs (can be very complex, this is a reasonable start)
    url_regex = re.compile(
        r'^(?:http|ftp)s?://' # http:// or https://
        r'(?:(?:[A-Z0-9](?:[A-Z0-9-]{0,61}[A-Z0-9])?\.)+(?:[A-Z]{2,6}\.?|[A-Z0-9-]{2,}\.?)|' # domain...
        r'localhost|' # localhost...
        r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3})' # ...or ip
        r'(?::\d+)?' # optional port
        r'(?:/?|[/?]\S+)$', re.IGNORECASE)
    if re.fullmatch(url_regex, url):
        return url
    else:
        print(f"Validation Error: '{url}' is not a valid URL.")
        return None

# --- Usage Examples ---
if __name__ == "__main__":
    print("\n--- Email Validation ---")
    email1 = validate_email("test@example.com")
    print(f"Validated email 'test@example.com': {email1}")

    email2 = validate_email("invalid-email")
    print(f"Validated email 'invalid-email': {email2}")

    print("\n--- Phone Number Validation ---")
    phone1 = validate_phone_number("123-456-7890")
    print(f"Validated phone '123-456-7890': {phone1}")

    phone2 = validate_phone_number("(123) 456-7890")
    print(f"Validated phone '(123) 456-7890': {phone2}")

    phone3 = validate_phone_number("1234567890")
    print(f"Validated phone '1234567890': {phone3}")

    phone4 = validate_phone_number("123-ABC-7890")
    print(f"Validated phone '123-ABC-7890': {phone4}")

    print("\n--- URL Validation ---")
    url1 = validate_url("https://www.google.com")
    print(f"Validated URL 'https://www.google.com': {url1}")

    url2 = validate_url("http://localhost:8080/api/data")
    print(f"Validated URL 'http://localhost:8080/api/data': {url2}")

    url3 = validate_url("ftp://example.org/file.zip")
    print(f"Validated URL 'ftp://example.org/file.zip': {url3}")

    url4 = validate_url("not-a-valid-url")
    print(f"Validated URL 'not-a-valid-url': {url4}")


### Deployment Setup for `RootAgent` to Vertex AI Agent Engine

To deploy your `RootAgent` as a Vertex AI Reasoning Engine, we need to add some boilerplate code for SDK installation, authentication, configuration, and a wrapper class to adapt your agent's interface to what `ReasoningEngine` expects.

In [91]:
# --- Step 1: Install necessary SDKs ---
# You might need to restart the runtime after this step if you're running this in a new Colab session.
!pip install --upgrade --quiet google-cloud-aiplatform[agent_engines,adk]>=1.112

# --- Step 2: Authenticate to Google Cloud ---
# This is often handled automatically or via a simple call in Colab Enterprise.
from google.colab import auth
auth.authenticate_user()

# --- Step 3: Import Vertex AI and initialize ---
import vertexai
from vertexai import agent_engines
from vertexai.preview import reasoning_engines

# Replace with your actual project ID and desired region
PROJECT_ID = "qwiklabs-gcp-04-994305c355a7"
REGION = "us-central1" # Or another supported region like 'us-east1', 'europe-west1'

# --- Step 6: Specify your Cloud Storage staging bucket ---
# Replace with an existing bucket name in your project, or create one.
STAGING_BUCKET = "gs://pp_agent-staging-bucket" # <<< IMPORTANT: CHANGE THIS TO AN ACTUAL GCS BUCKET NAME IN YOUR PROJECT!
print(f"Using staging bucket: {STAGING_BUCKET}")

vertexai.init(project=PROJECT_ID, location=REGION, staging_bucket=STAGING_BUCKET)


Using staging bucket: gs://pp_agent-staging-bucket


### Wrapper Class for `RootAgent`

The `ReasoningEngine` expects a `predict` method. We'll create a simple wrapper that initializes your `RootAgent` (which was already set up in a previous cell) and calls its `process_request` method from within the `predict` method.

In [90]:
from typing import Dict, Any

# Assuming RootAgent, WeatherAgent, InternetSearchAgent, GoogleMapsAgent,
# and QuestionAnsweringAgent are already defined and globally available from previous cells.

class RootAgentReasoningEngineApp:
    def __init__(self, project: str, location: str):
        # The sub-agents and RootAgent are already globally instantiated in the notebook
        # due to the previous refactoring. So, we don't need to re-instantiate them here.
        # We can directly reference the globally available 'root_agent' object.
        print(f"RootAgentReasoningEngineApp initialized. Project: {project}, Location: {location}")
        # We'll rely on the global 'root_agent' instance
        self.root_agent = globals().get('root_agent')
        if not self.root_agent:
            raise RuntimeError("RootAgent not found in global scope. Ensure it's initialized.")

    def query(self, **kwargs) -> Dict[str, Any]:
        # The 'kwargs' dictionary will now contain the user_input directly
        user_input = kwargs.get("user_input", "")

        if not user_input:
            return {"status": "failed", "result": "No user_input provided in the request.", "context_updates": {}}

        print(f"ReasoningEngineApp: Processing request: {user_input}")
        response = self.root_agent.process_request(user_input)
        return response

# Instantiate the wrapper app
root_agent_app = RootAgentReasoningEngineApp(project=PROJECT_ID, location=REGION)


RootAgentReasoningEngineApp initialized. Project: qwiklabs-gcp-04-994305c355a7, Location: us-central1


### Define Dependencies and Staging Bucket

List all Python libraries your agent code uses, and specify a Cloud Storage bucket for deployment staging.

In [83]:
AGENT_REQUIREMENTS = [
    "google-cloud-aiplatform[agent_engines,adk]>=1.112",
    "requests", # Used by WeatherAgent and for general API calls
    # "typing", # 'typing' is usually a built-in or implicitly handled but good to list if specific versions were needed
    # "re", # 're' is built-in
    # Add any other specific dependencies your agent needs
]

# After ensuring STAGING_BUCKET is correctly set, run the next cell to re-deploy the agent.


### Deploy the `RootAgent` Wrapper to Agent Engine

Now, we can use `vertexai.preview.reasoning_engines.ReasoningEngine.create` to deploy your agent.

In [92]:
print(f"Deploying agent 'RootAgent' to Agent Engine in {REGION}...")

# The vertexai.init call is now handled in the previous cell (befb8e56)
# This cell will now directly proceed with deployment, assuming vertexai is correctly initialized.

remote_root_agent = vertexai.preview.reasoning_engines.ReasoningEngine.create(
    root_agent_app,
    requirements=AGENT_REQUIREMENTS,
    display_name="MyCustomRootAgent", # A human-readable name for your deployed agent
    description="A custom RootAgent deployed from Colab Enterprise notebook.",
)

print(f"Deployment initiated. Agent Engine resource name: {remote_root_agent.name}")
print("This process can take several minutes. You can monitor its status in the Google Cloud Console under Vertex AI -> Agent Engine.")


INFO:vertexai.reasoning_engines._reasoning_engines:Using bucket pp_agent-staging-bucket


Deploying agent 'RootAgent' to Agent Engine in us-central1...


INFO:vertexai.reasoning_engines._reasoning_engines:Writing to gs://pp_agent-staging-bucket/reasoning_engine/reasoning_engine.pkl
INFO:vertexai.reasoning_engines._reasoning_engines:Writing to gs://pp_agent-staging-bucket/reasoning_engine/requirements.txt
INFO:vertexai.reasoning_engines._reasoning_engines:Creating in-memory tarfile of extra_packages
INFO:vertexai.reasoning_engines._reasoning_engines:Writing to gs://pp_agent-staging-bucket/reasoning_engine/dependencies.tar.gz
INFO:vertexai.reasoning_engines._reasoning_engines:Creating ReasoningEngine
INFO:vertexai.reasoning_engines._reasoning_engines:Create ReasoningEngine backing LRO: projects/818983390333/locations/us-central1/reasoningEngines/8471911914362896384/operations/7672733683010240512
INFO:vertexai.reasoning_engines._reasoning_engines:ReasoningEngine created. Resource name: projects/818983390333/locations/us-central1/reasoningEngines/8471911914362896384
INFO:vertexai.reasoning_engines._reasoning_engines:To use this ReasoningEng

Deployment initiated. Agent Engine resource name: 8471911914362896384
This process can take several minutes. You can monitor its status in the Google Cloud Console under Vertex AI -> Agent Engine.


In [ ]:
from typing import Any, Optional

def validate_required(value: Any, field_name: str) -> Optional[Any]:
    """
    Validates if a value is not None and not an empty string (after stripping whitespace).

    Args:
        value: The value to validate.
        field_name: The name of the field for error messages.

    Returns:
        The validated value if not empty, otherwise None.
    """
    if value is None:
        print(f"Validation Error: '{field_name}' is required but received None.")
        return None
    if isinstance(value, str) and not value.strip():
        print(f"Validation Error: '{field_name}' is required but received an empty string.")
        return None
    if isinstance(value, (list, dict, set)) and not value:
        print(f"Validation Error: '{field_name}' is required but received an empty collection.")
        return None
    return value

# --- Usage Examples ---
if __name__ == "__main__":
    print("\n--- Required Field Validation ---")
    name1 = validate_required("Alice", "Name")
    print(f"Validated 'Alice' for 'Name': {name1}")

    name2 = validate_required("", "Name")
    print(f"Validated '' for 'Name': {name2}")

    name3 = validate_required(None, "Name")
    print(f"Validated None for 'Name': {name3}")

    list1 = validate_required([1, 2], "Items")
    print(f"Validated '[1, 2]' for 'Items': {list1}")

    list2 = validate_required([], "Items")
    print(f"Validated '[]' for 'Items': {list2}")


In [ ]:
from typing import Optional

def validate_password_strength(password: str) -> Optional[str]:
    """
    Validates password strength (e.g., minimum length, complexity requirements).

    Args:
        password: The password string to validate.

    Returns:
        The validated password string if strong enough, otherwise None.
    """
    errors = []
    if len(password) < 8:
        errors.append("Password must be at least 8 characters long.")
    if not re.search(r"[A-Z]", password):
        errors.append("Password must contain at least one uppercase letter.")
    if not re.search(r"[a-z]", password):
        errors.append("Password must contain at least one lowercase letter.")
    if not re.search(r"\d", password):
        errors.append("Password must contain at least one digit.")
    if not re.search(r"[!@#$%^&*(),.?\":{}|<>]", password):
        errors.append("Password must contain at least one special character.")

    if errors:
        print(f"Validation Error: Password strength failed. {' '.join(errors)}")
        return None
    return password

# --- Usage Examples ---
if __name__ == "__main__":
    print("\n--- Custom Password Validation ---")
    password1 = validate_password_strength("MyStrongP@ss1")
    print(f"Validated password 'MyStrongP@ss1': {password1}")

    password2 = validate_password_strength("weakpass")
    print(f"Validated password 'weakpass': {password2}")

    password3 = validate_password_strength("NoDigitsHere!")
    print(f"Validated password 'NoDigitsHere!': {password3}")


In [ ]:
from typing import Optional

def validate_positive_integer(value_str: str) -> Optional[int]:
    """
    Validates if a string represents a positive integer (greater than 0).

    Args:
        value_str: The string input.

    Returns:
        The validated integer if positive, otherwise None.
    """
    try:
        num = int(value_str)
        if num > 0:
            return num
        else:
            print(f"Validation Error: '{value_str}' is not a positive integer.")
            return None
    except ValueError:
        print(f"Validation Error: '{value_str}' is not a valid integer.")
        return None

def validate_non_negative_float(value_str: str) -> Optional[float]:
    """
    Validates if a string represents a non-negative float (greater than or equal to 0).

    Args:
        value_str: The string input.

    Returns:
        The validated float if non-negative, otherwise None.
    """
    try:
        num = float(value_str)
        if num >= 0:
            return num
        else:
            print(f"Validation Error: '{value_str}' is negative.")
            return None
    except ValueError:
        print(f"Validation Error: '{value_str}' is not a valid number.")
        return None

# --- Usage Examples ---
if __name__ == "__main__":
    print("\n--- Numeric Input Validation ---")
    pos_int1 = validate_positive_integer("100")
    print(f"Validated '100' as positive integer: {pos_int1}")

    pos_int2 = validate_positive_integer("-10")
    print(f"Validated '-10' as positive integer: {pos_int2}")

    pos_int3 = validate_positive_integer("0")
    print(f"Validated '0' as positive integer: {pos_int3}")

    non_neg_float1 = validate_non_negative_float("123.45")
    print(f"Validated '123.45' as non-negative float: {non_neg_float1}")

    non_neg_float2 = validate_non_negative_float("-0.01")
    print(f"Validated '-0.01' as non-negative float: {non_neg_float2}")


In [ ]:
import re
from typing import Dict, List, Callable, Tuple, Any

class ValidationError(Exception):
    """Custom exception for validation errors."""
    def __init__(self, errors: Dict[str, List[str]]):
        self.errors = errors
        super().__init__(f"Validation failed with errors: {errors}")

class UserProfileValidator:
    """
    A validator class for user profile data.
    """
    def __init__(self):
        self._validators: Dict[str, List[Tuple[Callable[[Any], bool], str]]] = {
            "name": [
                (lambda x: isinstance(x, str) and len(x.strip()) > 0, "Name is required."),
                (lambda x: len(x) <= 50, "Name cannot exceed 50 characters.")
            ],
            "email": [
                (lambda x: isinstance(x, str), "Email must be a string."),
                (lambda x: re.fullmatch(r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$", x), "Invalid email format.")
            ],
            "age": [
                (lambda x: isinstance(x, int), "Age must be an integer."),
                (lambda x: 0 <= x <= 120, "Age must be between 0 and 120.")
            ],
            "is_active": [
                (lambda x: isinstance(x, bool), "Is_active must be a boolean.")
            ]
        }

    def validate(self, data: Dict[str, Any]) -> Dict[str, Any]:
        """
        Validates the entire data dictionary against predefined rules.

        Args:
            data: The dictionary containing the input data.

        Returns:
            The validated and potentially cleaned data dictionary.

        Raises:
            ValidationError: If any validation rule fails.
        """
        errors: Dict[str, List[str]] = {}
        cleaned_data: Dict[str, Any] = {}

        # Collect all expected fields to ensure required fields are checked
        all_expected_fields = set(self._validators.keys())

        for field in all_expected_fields:
            if field not in data:
                if field not in errors:
                    errors[field] = []
                errors[field].append(f"'{field}' is a required field.")
                continue # Skip validation for missing field

            value = data[field]
            field_has_errors = False # Flag to stop further validation for this field if an error occurs

            # Apply field-specific validators
            if field in self._validators:
                for validation_func, error_msg in self._validators[field]:
                    if not validation_func(value):
                        if field not in errors:
                            errors[field] = []
                        errors[field].append(error_msg)
                        field_has_errors = True
                        break # Stop validating this field if a rule fails

            # If no errors for this field, add to cleaned_data
            if not field_has_errors:
                cleaned_data[field] = value

        # Also handle any extra fields not defined in validators (optional, depending on strictness)
        for field, value in data.items():
            if field not in all_expected_fields:
                # Optionally, raise an error for unexpected fields or just ignore them
                # For now, we'll just add them to cleaned_data if no specific validation failed.
                if field not in errors: # Ensure it wasn't already caught as a required field
                    cleaned_data[field] = value

        if errors:
            raise ValidationError(errors)

        return cleaned_data

# --- Usage Examples ---
if __name__ == "__main__":
    print("\n--- Structured Validation with UserProfileValidator ---")
    validator = UserProfileValidator()

    # Valid data
    valid_data = {
        "name": "Jane Doe",
        "email": "jane.doe@example.com",
        "age": 30,
        "is_active": True
    }
    try:
        cleaned_valid_data = validator.validate(valid_data)
        print(f"Valid data processed: {cleaned_valid_data}")
    except ValidationError as e:
        print(f"Validation failed unexpectedly for valid data: {e.errors}")

    # Invalid data
    invalid_data = {
        "name": "Too Long Of A Name To Be Valid For This Profile Which Has A Fifty Character Limit",
        "email": "bad-email",
        "age": "thirty", # Wrong type
        "is_active": "yes" # Wrong type
        # 'phone' field is missing and not explicitly handled by this validator,
        # would typically be added to self._validators
    }
    try:
        validator.validate(invalid_data)
    except ValidationError as e:
        print(f"Invalid data errors: {e.errors}")

    # Data with missing required field
    missing_data = {
        "name": "John Doe",
        "email": "john.doe@example.com",
        "is_active": False
    }
    try:
        validator.validate(missing_data)
    except ValidationError as e:
        print(f"Missing field errors: {e.errors}")

# Task
Create a new Google Cloud Storage bucket named `pp_agent-staging-bucket`, copy all data from the existing bucket `gs://your-colab-agent-staging-bucket` to it, and then verify the new bucket's creation and data population.

## Create New GCS Bucket

### Subtask:
Create a new Google Cloud Storage bucket with the name `pp_agent-staging-bucket`.


**Reasoning**:
The subtask requires creating a new Google Cloud Storage bucket. I will use the `gsutil mb` command in a code block to create the bucket with the specified name.



In [77]:
BUCKET_NAME = 'pp_agent-staging-bucket'
!gsutil mb gs://{BUCKET_NAME}
print(f"Bucket 'gs://{BUCKET_NAME}' created (or already exists).")

Creating gs://pp_agent-staging-bucket/...
Bucket 'gs://pp_agent-staging-bucket' created (or already exists).


## Copy Bucket Contents

### Subtask:
Copy all contents from the existing bucket `gs://your-colab-agent-staging-bucket` to the newly created bucket `gs://pp_agent-staging-bucket`.


**Reasoning**:
The subtask requires copying all contents from the source bucket to the destination bucket. I will use the `gsutil cp -r` command to perform this recursive copy.



In [78]:
SOURCE_BUCKET = STAGING_BUCKET
DESTINATION_BUCKET = BUCKET_NAME

!gsutil cp -r {SOURCE_BUCKET}/* gs://{DESTINATION_BUCKET}/

print(f"All contents from '{SOURCE_BUCKET}' copied to 'gs://{DESTINATION_BUCKET}/'.")

Copying gs://your-colab-agent-staging-bucket/reasoning_engine/dependencies.tar.gz [Content-Type=text/plain]...
Copying gs://your-colab-agent-staging-bucket/reasoning_engine/reasoning_engine.pkl [Content-Type=application/octet-stream]...
Copying gs://your-colab-agent-staging-bucket/reasoning_engine/requirements.txt [Content-Type=text/plain]...
- [3 files][ 24.0 KiB/ 24.0 KiB]                                                
Operation completed over 3 objects/24.0 KiB.                                     
All contents from 'gs://your-colab-agent-staging-bucket' copied to 'gs://pp_agent-staging-bucket/'.


**Reasoning**:
The previous step copied the data. Now, I need to verify that the data has been successfully copied to the new bucket by listing its contents.



In [79]:
print(f"Listing contents of 'gs://{DESTINATION_BUCKET}/':")
!gsutil ls -r gs://{DESTINATION_BUCKET}/


Listing contents of 'gs://pp_agent-staging-bucket/':
gs://pp_agent-staging-bucket/reasoning_engine/:
gs://pp_agent-staging-bucket/reasoning_engine/dependencies.tar.gz
gs://pp_agent-staging-bucket/reasoning_engine/reasoning_engine.pkl
gs://pp_agent-staging-bucket/reasoning_engine/requirements.txt


## Final Task

### Subtask:
Confirm that the new bucket `pp_agent-staging-bucket` has been created and populated with the copied data.


## Summary:

### Data Analysis Key Findings
*   A new Google Cloud Storage (GCS) bucket named `pp_agent-staging-bucket` was successfully created.
*   Three objects, totaling 24.0 KiB, were copied from the source bucket `gs://your-colab-agent-staging-bucket` to the new bucket `gs://pp_agent-staging-bucket`.
*   The copied files include `reasoning_engine/dependencies.tar.gz`, `reasoning_engine/reasoning_engine.pkl`, and `reasoning_engine/requirements.txt`.
*   Verification confirmed that all three copied files are present in the destination bucket `gs://pp_agent-staging-bucket`.

### Insights or Next Steps
*   The `pp_agent-staging-bucket` is now fully set up and populated with the necessary data, ready for staging purposes.
*   This successful migration ensures data availability and integrity in the new staging environment.
